In [0]:
%run ./01-config

In [0]:
import time
from pyspark.sql import functions as F

# --- Configuration ---
spark.sql(f"USE {catalog}.{db_name}")
start = int(time.time())
print("Starting gold layer batch aggregation...")

# ==============================================================================
# GOLD LAYER: Workout BPM Summary
# Aggregates heart rate per workout session, enriched with user demographics.
# ==============================================================================

# --- Workout BPM Summary ---
print("Upserting workout_bpm_summary...", end='')

# Load demographic bins for enrichment
df_user_bins = spark.read.table(f"{catalog}.{db_name}.user_bins")

# Aggregate heart rate stats per workout session
df_summary = (spark.read.table(f"{catalog}.{db_name}.workout_bpm")
    .groupBy("user_id", "workout_id", "session_id")
    .agg(
        F.min("heartrate").alias("min_bpm"),
        F.mean("heartrate").alias("avg_bpm"),
        F.max("heartrate").alias("max_bpm"),
        F.count("heartrate").alias("num_recordings")
    )
    .join(df_user_bins, ["user_id"])
    .select(
        "workout_id", "session_id", "user_id",
        "age", "gender", "city", "state",
        "min_bpm", "avg_bpm", "max_bpm", "num_recordings"
    )
)

df_summary.createOrReplaceTempView("workout_bpm_summary_delta")
spark.sql(f"""
    MERGE INTO {catalog}.{db_name}.workout_bpm_summary a
    USING workout_bpm_summary_delta b
    ON a.user_id = b.user_id AND a.workout_id = b.workout_id AND a.session_id = b.session_id
    WHEN NOT MATCHED THEN INSERT *
""")
print("Done")

print(f"\nGold layer batch aggregation completed in {int(time.time()) - start} seconds")
    


In [0]:
# ==============================================================================
# GOLD LAYER VALIDATION (Batch)
# Set `sets` to 1 for a single batch load, 2 for double batch
# ==============================================================================
import time

sets = 1  # <-- UPDATE: 1 for single batch, 2 for double batch
test_data_dir = base_dir_data + "/test_data"  # <-- UPDATE if your test parquet files are elsewhere

start_val = int(time.time())
print("Starting Gold layer validation...\n")

# --- Validate gym_summary (exact row-by-row comparison against expected Parquet) ---
print("Validating records in gym_summary...", end='')
expected_rows = spark.read.format("parquet").load(f"{test_data_dir}/7-gym_summary_{sets}.parquet").collect()
actual_rows = spark.read.table(f"{catalog}.{db_name}.gym_summary").collect()
assert expected_rows == actual_rows, (
    f"\n Data mismatch in gym_summary\n"
    f"- Expected: {len(expected_rows)} rows\n"
    f"- Actual:   {len(actual_rows)} rows"
)
print(f" Expected data matches actual data in gym_summary: OK")

# --- Validate workout_bpm_summary (count check for multi-set scenarios) ---
if sets > 1:
    print("Validating record counts in workout_bpm_summary...", end='')
    expected_count = 3
    actual_count = spark.read.table(f"{catalog}.{db_name}.workout_bpm_summary").count()
    assert actual_count == expected_count, \
        f"Expected {expected_count:,} records, found {actual_count:,} in workout_bpm_summary"
    print(f" {actual_count:,} / {expected_count:,} records - OK")

print(f"\nGold layer validation completed in {int(time.time()) - start_val} seconds")